# 05 – Comparison of methods

**Purpose:** compare every approach on the brief's criteria (report Section 5): accuracy, robustness, computational time, memory use, implementation complexity, interpretability, scalability and suitability for HVAC scheduling.

| Approach | What it is |
|---|---|
| Fixed timetable | HVAC on 8 am–8 pm weekdays; no forecast |
| Profile baseline | Average headcount for each space × weekday × hour |
| Linear regression | Weighted sum of the same inputs as the ML model (one-hot hour/weekday/space, semester, recent occupancy) |
| Random forest | Many deep trees grown independently and averaged |
| **Gradient boosting (selected)** | Small trees built in sequence, each correcting the last |

All learning approaches use the same inputs and the same chronological split: trained before 2025 and scored on 2025, then retrained to the end of 2025 and scored on Jan–Apr 2026. Settings for linear regression and the random forest were not tuned; the gradient-boosting settings come from notebook 03.

In [1]:
import functools
import numpy as np
import pandas as pd
from occupancy import clean, evaluate, features, robustness
from occupancy.models import (GradientBoostingForecaster, LinearRegressionForecaster, RandomForestForecaster,
                              ProfileBaseline, FixedTimetable, TUNED_SETTINGS)
from occupancy.config import PROCESSED_DIR, REPO_ROOT

RESULTS = REPO_ROOT / "reports" / "results"
data = features.add_features(clean.load_hourly(PROCESSED_DIR / "occupancy_hourly.csv"))
split = evaluate.chronological_split(data)
train, val, test = split["train"], split["validation"], split["test"]
history = pd.concat([train, val])

MODELS = {  # picklable factories, so each can be timed in a fresh process
    "Profile baseline": ProfileBaseline,
    "Linear regression": LinearRegressionForecaster,
    "Random forest": RandomForestForecaster,
    "Gradient boosting": functools.partial(GradientBoostingForecaster, **TUNED_SETTINGS),
}
timetable = FixedTimetable(8, 20)

## 1. Accuracy (2025 validation and 2026 test)

In [2]:
preds = {"2025": {}, "2026": {}}
rows = []
for period, tr, d in [("2025", train, val), ("2026", history, test)]:
    rows.append({"period": period, **evaluate.score_schedule(d, timetable.schedule(d), "Fixed timetable")})
    for name, make in MODELS.items():
        preds[period][name] = make().fit(tr).predict(d)
        rows.append({"period": period, **evaluate.score_forecast(d, preds[period][name], name)})
accuracy = pd.DataFrame(rows)
accuracy.to_csv(RESULTS / "stage5_accuracy.csv", index=False)
accuracy.pivot(index="approach", columns="period", values=["mae", "rmse", "accuracy", "wasted", "discomfort"]) \
    .reindex(["Fixed timetable"] + list(MODELS)).round(3)

mae          rmse        accuracy         wasted          \
period              2025   2026   2025   2026     2025   2026    2025    2026   
approach                                                                        
Fixed timetable      NaN    NaN    NaN    NaN    0.752  0.763  3339.0  1016.0   
Profile baseline   0.753  0.678  1.368  1.381    0.727  0.747  4927.0  1259.0   
Linear regression  0.640  0.626  1.256  1.316    0.753  0.762  4439.0  1172.0   
Random forest      0.605  0.595  1.247  1.290    0.772  0.783  3640.0   930.0   
Gradient boosting  0.595  0.581  1.243  1.283    0.780  0.792  3370.0   836.0   

                  discomfort         
period                  2025   2026  
approach                             
Fixed timetable       2807.0  681.0  
Profile baseline      1821.0  549.0  
Linear regression     1667.0  530.0  
Random forest         2015.0  623.0  
Gradient boosting     2065.0  651.0

## 2. Robustness
Is gradient boosting's lead over linear regression (the closest simple alternative) real? Bootstrap by day as in notebook 04, and month-by-month comparison over 2025–2026.

In [3]:
boot = pd.DataFrame({p: robustness.bootstrap_mae_difference(d, preds[p]["Gradient boosting"], preds[p]["Linear regression"])
                     for p, d in [("2025", val), ("2026", test)]}).T
boot.to_csv(RESULTS / "stage5_bootstrap_gb_vs_linear.csv")
boot.round(3)

,difference,ci_low,ci_high,share_a_better
2025,-0.046,-0.055,-0.037,1.0
2026,-0.045,-0.057,-0.032,1.0


In [4]:
both = pd.concat([val, test])
months = both["date"].str[:7]
monthly = pd.DataFrame({name: pd.concat([preds["2025"][name], preds["2026"][name]]) for name in MODELS})
monthly_mae = monthly.sub(both["mean_headcount"], axis=0).abs().groupby(months).mean()
best = monthly_mae.idxmin(axis=1).value_counts().rename("months with lowest MAE (of 16)")
monthly_mae.to_csv(RESULTS / "stage5_monthly_mae.csv")
print(best.to_string())
print("\nWorst month MAE for each approach:")
monthly_mae.max().round(3)

Gradient boosting    16

Worst month MAE for each approach:


Profile baseline     0.843
Linear regression    0.765
Random forest        0.711
Gradient boosting    0.695
dtype: float64

## 3. Computational time and memory
Each model is trained on all data up to the end of 2025 (54,000 hours) in a fresh Python process, so its memory use is measured on its own. *Training memory* is the extra peak memory needed while fitting; *stored model* is the size of the saved model a building system would keep.

In [5]:
day = test[test["date"] == "2026-03-10"]
cost = pd.DataFrame({name: robustness.fit_cost(make, history, day) for name, make in MODELS.items()}).T
cost.to_csv(RESULTS / "stage5_cost.csv")
cost.round(4)

,fit_seconds,forecast_one_day_seconds,training_memory_mb,stored_model_mb
Profile baseline,0.0152,0.0037,10.3906,0.0158
Linear regression,0.1924,0.0066,32.9766,0.0031
Random forest,17.0500,0.0629,98.5586,73.0080
Gradient boosting,1.5982,0.0093,31.6953,1.0832


## 4. Scalability
How does training time grow with the amount of data? Each model is trained on 25 %, 50 % and 100 % of the history.

In [6]:
rows = []
for frac in [0.25, 0.5, 1.0]:
    sub = history.iloc[: int(len(history) * frac)]
    for name, make in MODELS.items():
        rows.append({"training hours": len(sub), "approach": name,
                     "fit seconds": robustness.fit_cost(make, sub, day, repeats=1)["fit_seconds"]})
scaling = pd.DataFrame(rows).pivot(index="training hours", columns="approach", values="fit seconds")
scaling.to_csv(RESULTS / "stage5_scaling.csv")
growth = (scaling.iloc[-1] / scaling.iloc[0]).rename("time ratio, 4x data")
pd.concat([scaling.round(3), growth.to_frame().T.round(2)])

approach,Gradient boosting,Linear regression,Profile baseline,Random forest
13457,0.795,0.157,0.009,3.614
26914,1.208,0.158,0.011,7.944
53828,1.682,0.189,0.015,17.475
"time ratio, 4x data",2.120,1.210,1.570,4.840


## 5. Interpretability
The linear model can be read directly: each input has one weight (people added per unit). The tree models have no single formula; notebook 03 explains them with permutation importance instead.

In [7]:
lin = LinearRegressionForecaster().fit(history)
names = lin.model_[0].get_feature_names_out()
coef = pd.Series(lin.model_[-1].coef_, index=names)
readable = coef[[n for n in names if not n.startswith("onehotencoder")]]
readable.index = [n.split("__")[1] for n in readable.index]
print(f"{len(coef)} weights in total (most are one-hot hour / weekday / space offsets). Non-calendar weights:")
readable.round(3)

45 weights in total (most are one-hot hour / weekday / space offsets). Non-calendar weights:


in_semester                        0.286
lag_1d                             0.130
lag_7d                             0.119
mean_4w                            0.194
yesterday_mean                     0.243
missingindicator_lag_1d           -0.199
missingindicator_lag_7d           -0.183
missingindicator_mean_4w           0.078
missingindicator_yesterday_mean    0.248
dtype: float64

## 6. Summary
* **Accuracy.** Gradient boosting has the lowest error in both periods (MAE 0.595 / 0.581 people) and the highest share of correct HVAC decisions (78.0 % / 79.2 %). The random forest is close behind (0.605 / 0.595). Linear regression, given the same inputs, beats the profile baseline (0.640 vs 0.753 in 2025) but not the tree models, because it cannot learn interactions. It has the fewest discomfort hours of the forecasts, but at the cost of many more wasted hours.
* **Robustness.** Gradient boosting's lead over linear regression is small but consistent: about 0.045 people, with a 95 % bootstrap interval entirely below zero in both periods, and the lowest MAE of the four forecasts in all 16 months. Its worst month (0.695) is better than the other forecasts' worst months.
* **Time and memory.** All forecasts take a hundredth of a second or less for a whole day. Training takes about 1.5 s for gradient boosting, 0.2 s for linear regression and 17 s for the random forest. Gradient boosting stores a 1 MB model and needs about 30 MB extra while training; the random forest stores 73 MB and needs about 100 MB.
* **Scalability.** With 4× the data, training time rose by less than 2× for gradient boosting and linear regression, but by about 4.5× for the random forest (it grows full-depth trees on every hour).
* **Interpretability.** Linear regression is the easiest to read: one weight per input (e.g. a semester weekday adds about 0.29 people, and each extra person yesterday at the same hour adds about 0.13). Gradient boosting has no single formula and has to be explained with permutation importance (notebook 03). The profile baseline and timetable are fully transparent.
* **Conclusion.** Gradient boosting gives the best accuracy at a small, practical cost, which justifies selecting it. Linear regression is the best fallback if transparency matters more than about 0.05 people of extra error.
